## Augmentation

Augmentation performed over every image of dataset and created 10 augment per image

In [1]:
import cv2
import numpy as np
import os
from pathlib import Path
import random


In [ ]:
INPUT_DIR = r"C:\Users\singh\Users\singh\Desktop\Palm detection\new palm"
OUTPUT_DIR = r"C:\Users\singh\Users\singh\Desktop\Palm detection\Research paper\01_data 5"
AUG_PER_IMAGE = 10 

In [ ]:


def safe_brightness(img, low=-30, high=30):
    """Adjusts brightness via HSV to preserve texture detail."""
    value = random.randint(low, high)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    if value >= 0:
        v = cv2.add(v, value) 
    else:
        v = cv2.subtract(v, abs(value))
        
    return cv2.cvtColor(cv2.merge((h, s, v)), cv2.COLOR_HSV2BGR)

def safe_rotate(img, angle_limit=5):
    """Slight rotation only—palmprints are usually fairly aligned."""
    angle = random.uniform(-angle_limit, angle_limit)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    # BORDER_REPLICATE avoids introducing artificial black edges/grids
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

def run_augmentation():
    src = Path(INPUT_DIR)
    dst = Path(OUTPUT_DIR)
    extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

    if not src.exists():
        print(f"Error: Input directory {INPUT_DIR} not found.")
        return

    for img_path in src.rglob('*'):
        if img_path.suffix.lower() in extensions:
            rel_path = img_path.relative_to(src).parent
            target_folder = dst / rel_path
            target_folder.mkdir(parents=True, exist_ok=True)

            img = cv2.imread(str(img_path))
            if img is None: continue

            # Save original
            cv2.imwrite(str(target_folder / f"orig_{img_path.name}"), img)

            # Perform non-destructive Augmentation
            for i in range(AUG_PER_IMAGE):
                temp_img = img.copy()

                # 1. Subtle Brightness shift (Simulates different lighting)
                temp_img = safe_brightness(temp_img)

                # 2. Very slight Rotation (Simulates minor hand tilt)
                temp_img = safe_rotate(temp_img)

                # Note: NO FLIPPING performed here to preserve hand handedness

                save_path = target_folder / f"aug_{i}_{img_path.name}"
                cv2.imwrite(str(save_path), temp_img)

    print(f"Done! Palmprint-safe augmented dataset is at: {OUTPUT_DIR}")

if __name__ == "__main__":
    run_augmentation()

Done! Palmprint-safe augmented dataset is at: C:\Users\singh\Users\singh\Desktop\Palm detection\Research paper\01_data 5
